---
# Chapter 13 — When the Frame Is Wrong

## Orientation

| Field | Value |
|-------|-------|
| Chapter | 13: When the Frame Is Wrong |
| Central question | How does the system establish the correct frame, and what happens when it's wrong? |
| Main concepts | Frame establishment, Reconciliation, Safety boundary, Breach-free policy |
| Implementation | context_frames |
| Experiment | ch13-eval-v1-ministral |
| Evidence status | Book result: conditional control |
| Depends on | Chapter 10 (frames), Chapter 12 (behavioural hinge) |

---

## What this notebook demonstrates

Chapter 10 showed that frames are powerful but hazardous — wrong or inferred frames can destroy recall. Chapter 13 resolves this hazard via **reconciliation-derived establishment**. The notebook:

1. **Loads the frozen frame establishment run** (`ch13-eval-v1-ministral`)
2. **Shows the reconciliation gate** — the only breach-free policy on both readers
3. **Demonstrates a case where the wrong frame is harmful** and how the earned gate changes what influences behaviour
4. **Shows why per-reader simplifications disagree** and neither transfers

> **Evidence status**: Book result. The chapter reports reconciliation-derived classes matching 9/9 on two readers with no breaches. This notebook loads the frozen eval run (promotion gates: `promoted=True`, `breaches=[]`) and reproduces the gate locally: on three scenario classes the retrieved action agrees with the ledger even where the class label differs (CONFLICTING vs STALE both refuse the frame). Per-reader simplifications disagree and neither transfers, so the full gate survives only as the breach-free policy on both.

## The chapter question

> **What happens when the present frame is wrong?**

Chapter 10 earned explicit framing as a control mechanism but showed inferred frames are hazardous. Chapter 13 asks: how do we establish the frame safely?

## Concepts in this chapter

In [ ]:
import sys
from pathlib import Path

def _find_repo_root(start):
    cur = Path(start).resolve()
    while True:
        if ((cur / "content").is_dir() and (cur / "notebooks").is_dir()
                and (cur / "solution").is_dir()):
            return cur
        if cur == cur.parent:
            raise RuntimeError("could not locate repository root")
        cur = cur.parent

REPO_ROOT = _find_repo_root(Path.cwd())
for _p in (str(REPO_ROOT), str(REPO_ROOT / "solution")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from notebooks.memory._support import load_chapter_metadata, render_table

meta = load_chapter_metadata(13)
concepts = (meta.get('chapter', {}).get('concepts')
            or meta.get('concepts', []))
render_table([
    {"Concept ID": c['id'], "Name": c['name'], "Status": c['status']}
    for c in concepts
], "Chapter 13 Concepts")

## Load the frozen frame establishment run

In [ ]:
from notebooks.memory._support import load_frozen_run

run = load_frozen_run("ch13-eval-v1-ministral")
metrics = run["metrics"]

print(f"Run ID: {run['run_id']}")
print("\nMean task success by frame condition:")
for cond, stats in metrics["summary"]["mean_by_condition"].items():
    print(f"  {cond}: n={stats['n']} success={stats['task_success_macro']} "
          f"tokens={stats['mean_tokens']}")
gates = metrics["promotion_gates"]
print(f"\nPromotion gates: promoted={gates['promoted']} "
      f"breaches={gates['breaches']}")

## The reconciliation gate

The frame establishment policy that earned its place:

```text
Reconciliation-derived establishment:
  1. Propose frame from query + context
  2. Check against declared ProjectFrame/WorkFrame
  3. Check against temporal state (supersession)
  4. Check against evidence lineage (support)
  5. Check against open-loop status
  6. If ALL checks pass → HARD frame (enforced)
  7. If ANY check fails → SOFT frame (broaden toward strong RAG)
```

This is the **only breach-free policy on both readers** — per-reader simplifications disagree and neither transfers.

In [ ]:
# The staged establishment gate on frozen fixture scenarios: a declared
# frame goes HARD, a weakly-evidenced frame goes SOFT, a superseded
# frame is refused query-only retrieval. No model calls.
from context_frames import fixtures as FX
from context_frames import frame_fixtures as FF
from context_frames import frame_safety as FS
from context_frames.frames import keyword_work_frame

for sid in ("A-fix-dec-dev", "B-fix-weak-eval", "C-arch-shift-eval"):
    sc = FF.scenario_by_id(sid)
    dec, tr = FS.classify_establishment(sc.signals, FX.PF_MEMORY_BOOK(),
                                        FX.AS_OF, ())
    cond = FS.condition_for_action(dec.action)
    print(f"{sid}: expected={sc.expected_class}/{sc.expected_action}")
    print(f"  got: {dec.establishment}/{dec.action} "
          f"reasons={list(dec.reasons)}")
    print(f"  retrieval condition: {cond.name} (soft={cond.soft_frame}, "
          f"flat={cond.flat_rank})")
    print(f"  action match: {dec.action == sc.expected_action}")

declared = FX.WF_T1_PUBLICATION()
inferred = keyword_work_frame("wf-keyword-demo", FX.PF_MEMORY_BOOK(),
                              declared.signals, FX.AS_OF)
print(f"\ndeclared frame: {declared.work_type} [{declared.derivation}]")
print(f"keyword-inferred frame: {inferred.work_type} [{inferred.derivation}]")
print(f"unprovenanced fields: {inferred.unprovenanced_fields()}")

## The harmful wrong frame case

Chapter 10 showed: **a wrong frame could be worse than no frame**. Let's demonstrate:

In [ ]:
# The wrong-frame hazard, measured: the same publication task under
# HARD (C5), SOFT (C5-soft, exclusion disabled) and QUERY_ONLY (C0).
# MUST-ledger coverage is the score; the frame decides what survives.
from context_frames import build_context
from context_frames import corpus as CORPUS
from context_frames.retrieval import HybridRetriever
from context_frames.policy import CONDITIONS

units = CORPUS.by_id()
retriever = HybridRetriever(CORPUS.corpus())
t_pub = FX.T1_PUBLICATION()
must = {u for u, g in t_pub.ledger.items() if g == "MUST"}

for name, cond in [("C5 hard", CONDITIONS["C5"]),
                   ("C5-soft (inferred frames)",
                    FS.condition_for_action("SOFT_FRAME")),
                   ("C0 query-only", CONDITIONS["C0"])]:
    bundle, trace = build_context(cond, t_pub.query, FX.PF_MEMORY_BOOK(),
                                  t_pub.work_frame, retriever, units,
                                  budget_tokens=1500)
    got = {i.item_id.replace("item-", "") for i in bundle.items}
    print(f"{name:26s} admitted={len(bundle.items):2d} "
          f"MUST covered={len(must & got)}/{len(must)}")

## Why per-reader simplifications fail

The chapter found that **neither per-reader simplification transfers** — the full reconciliation gate is the only breach-free policy:

In [ ]:
# Show the failed simplifications
simplifications = [
    {"Simplification": "Skip temporal check", "Reader A": "Passes", "Reader B": "Fails (breach)", "Transfers": "No"},
    {"Simplification": "Skip evidence check", "Reader A": "Fails (breach)", "Reader B": "Passes", "Transfers": "No"},
    {"Simplification": "Skip open-loop check", "Reader A": "Passes", "Reader B": "Passes", "Transfers": "No (untested class)"},
    {"Simplification": "Use only ProjectFrame", "Reader A": "Fails (cross-project leakage)", "Reader B": "Fails", "Transfers": "No"},
]
render_table(simplifications, "Per-Reader Simplifications: Neither Transfers (Chapter 13 Finding)")

## What this establishes

- **Reconciliation-derived establishment is the only breach-free policy** on both readers
- **Per-reader simplifications disagree and neither transfers** — the full gate is necessary
- **Inferred frames stay conditional control, never core architecture**
- **The safety boundary is earned** — not assumed
- **Frame establishment itself is a hazard** that must be controlled

## What this does NOT establish

- Frame establishment works perfectly (it's conditional)
- Real-corpus frame reconciliation quality (untested)
- Multi-agent frame disagreement (untested)

## Try it yourself

Create a proposed frame that passes some reconciliation checks but fails others. Observe how the gate responds.

In [ ]:
# TRY IT YOURSELF: the stale-shift scenario. A newer DIRECT signal
# proposes different work, so the prior frame is STALE and retrieval
# falls back to query-only rather than trusting it.
sc = FF.scenario_by_id("C-arch-shift-eval")
dec, tr = FS.classify_establishment(sc.signals, FX.PF_MEMORY_BOOK(),
                                    FX.AS_OF, ())
print(f"stale-shift scenario: {dec.establishment} -> {dec.action}")
print(f"reasons: {list(dec.reasons)}")
bundle, trace = build_context(
    FS.condition_for_action(dec.action), t_pub.query, FX.PF_MEMORY_BOOK(),
    t_pub.work_frame, retriever, units, budget_tokens=1500)
print(f"QUERY_ONLY bundle: {len(bundle.items)} items "
      f"(the frame contributes no selection)")

## Where this leads next

Chapter 14 completes another separation: **Context is a Bottleneck** — memory is durable, context is selected.

> **See this chapter in code:** [Open the companion Jupyter notebook](memory\13-chapter.ipynb)